## 주식 데이터 수집 및 전처리

- `yfinance` 라이브러리를 사용하여 주식 데이터를 수집합니다. (2020-01-01 ~ 현재)
- 날짜 인덱스와 결측 개념을 확인합니다.
- 일간 수익률과 누적 수익률을 계산합니다.
- 종가 기준 가격 수익률 분석 결과를 다양한 그래프로 시각화합니다.

### 필요한 모듈 임포트

In [ ]:
# Colab에 yfinance 라이브러리 설치 여부를 확인합니다.
# [참고] pip show는 설치된 라이브러리의 정보를 콘솔에 출력합니다.
# 만약 해당 라이브러리가 설치되지 않았으면 Package(s) not found 경고를 출력합니다.
!pip show yfinance

In [ ]:
# yfinance 라이브러리를 설치합니다.
# [참고] 위 코드를 실행한 결과 Name, Version 등의 패키지 정보가 출력되었으면 설치 코드는 생략합니다.
!pip install yfinance

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import yfinance as yf

### 주식 데이터 수집

In [ ]:
# 티커(종목코드)를 설정합니다.
# [참고] 유가증권시장(KOSPI) 종목은 여섯 자리 종목코드 뒤에 '.KS'를 추가합니다. (코스닥은 '.KQ')
ticker = '005930.KS'

In [ ]:
# 주식 데이터 수집 시작일자를 설정합니다.
# [참고] 단순 수익률을 계산할 때 첫 번째 행은 결측이 됩니다.
start_date = '2020-01-01'

In [ ]:
# 주식 데이터 수집 종료일자를 설정합니다.
# [참고] 현재 시스템 날짜시간을 'yyyy-mm-dd' 형태의 문자열로 변환해 end_date에 할당합니다.
end_date = pd.Timestamp.today().strftime(format='%Y-%m-%d')

In [ ]:
# end_date를 확인합니다.
end_date

In [ ]:
# 주식 데이터를 수집하고 데이터프레임을 생성합니다.
# [참고] end 매개변수에 지정한 날짜를 포함하지 않습니다.
# auto_adjust 매개변수에 False를 지정하면 조정되지 않은 OHLC를 유지하고 수정 종가를 함께 반환합니다.(기본값: True)
# progress 매개변수에 False를 지정하면 다운로드 진행 표시를 출력하지 않습니다.(기본값: True)
# multi_level_index 매개변수에 False를 지정하면 단일 레벨 열 이름을 반환합니다.(기본값: True)
df = yf.download(
    tickers=ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
    multi_level_index=False,
)

### 데이터프레임 확인

In [ ]:
# df의 처음 5행을 확인합니다.
# [참고] n 매개변수에 출력할 행 개수를 정수로 지정합니다.(기본값: 5)
df.head()

In [ ]:
# df의 컬럼스(열 이름)를 확인합니다.
# [참고] 열 이름은 Adj Close(수정 종가), Close(종가), High(고가), Low(저가), Open(시가), Volume(거래량) 순입니다.
df.columns

In [ ]:
# df의 인덱스(행 이름)를 확인합니다.
# [참고] 인덱스 자료형은 DatetimeIndex입니다.
df.index

In [ ]:
# df의 정보를 확인합니다.
# [참고] 행 개수, 열 개수, 열 이름, 열별 결측값 아닌 개수 및 열별 자료형을 출력합니다.
df.info()

### 열 선택 및 순서 변경

In [ ]:
# df에서 Adj Close 열을 삭제합니다.
df = df.drop(columns='Adj Close')

In [ ]:
# df의 열 순서를 변경하기 위해 열 이름 리스트를 생성합니다.
cols = ['Open', 'High', 'Low', 'Close', 'Volume']

In [ ]:
# df의 열 순서를 변경합니다.
df = df.loc[:, cols]

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### 지저분한 데이터 만들기

In [ ]:
# df의 인덱스를 초기화하고 기존 인덱스를 첫 번째 열로 삽입합니다.
df = df.reset_index()

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df에서 Date를 문자열로 변환합니다.
df['Date'] = df['Date'].astype(dtype=str)

In [ ]:
# df의 열별 자료형을 확인합니다.
df.dtypes

In [ ]:
# df의 행 순서를 무작위로 섞어서 시간 순서를 망가뜨립니다.
# [참고] 매번 같은 결과를 얻기 위해 시드를 고정합니다.
df = df.sample(frac=1, random_state=1)

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df의 인덱스를 초기화하고, 기존 인덱스를 삭제합니다.
df = df.reset_index(drop=True)

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### 날짜 인덱스 전처리

In [ ]:
# df에서 Date를 날짜시간형으로 변환합니다.
df['Date'] = pd.to_datetime(arg=df['Date'])

In [ ]:
# df에서 Date를 인덱스로 설정합니다.
df = df.set_index(keys='Date')

In [ ]:
# df를 인덱스 기준으로 오름차순 정렬합니다.
df = df.sort_index()

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### [참고] 누락된 인덱스 추가 및 결측값 처리

- 주식 데이터는 거래일에만 값이 존재하므로, 주말 또는 공휴일의 인덱스는 누락되어 있습니다.
  - 시계열 데이터프레임에서 누락된 인덱스를 추가하면 해당일은 결측값(`np.nan`)으로 채워집니다.
- 연속적인 시계열 데이터는 누락된 인덱스를 추가하고 선형 보간법 등을 사용하여 결측을 채울 수 있습니다.
- [주의] 아래 코드는 결측값 처리 메서드의 사용법을 확인하기 위한 예시이며, 실제 비거래일의 거래량을 채우는 용도로 사용하지 않습니다.

In [ ]:
# df의 인덱스를 일(day) 단위로 재생성하고, 처음 5행만 선택하여 df_na에 할당합니다.
df_na = df.asfreq(freq='D').head()

In [ ]:
# df_na를 확인합니다.
df_na

In [ ]:
# df_na에서 거래량의 결측값을 직전 값으로 채운 결과를 확인합니다.
df_na['Volume'].ffill()

In [ ]:
# df_na에서 거래량의 결측값을 선형 보간법으로 채운 결과를 확인합니다.
df_na['Volume'].interpolate(method='linear')

### [참고] 전역 변수 목록 확인 및 변수 삭제

In [ ]:
# 전역 변수 목록을 확인합니다.
%whos

In [ ]:
# 전역 변수 목록에서 데이터프레임만 확인합니다.
%whos DataFrame

In [ ]:
# df_na를 전역 변수 목록에서 삭제합니다.
del df_na

In [ ]:
# 전역 변수 목록에서 데이터프레임만 확인합니다.
%whos DataFrame

### 날짜 인덱스 필터링

In [ ]:
# df에서 2020년인 행을 선택합니다.(인덱싱)
df.loc['2020']

In [ ]:
# df에서 2020년 1월인 행을 선택합니다.(인덱싱)
df.loc['2020-01']

In [ ]:
# df에서 2020년 1월 2일, 3일인 행을 선택합니다.(팬시 인덱싱)
df.loc[['2020-01-02', '2020-01-03']]

In [ ]:
# df에서 2020년 1월 1일부터 7일까지 연속된 행을 선택합니다.(슬라이싱)
df.loc['2020-01-01':'2020-01-07']

### 날짜 인덱스의 속성 활용

In [ ]:
# df에서 매년 1월인 행을 선택합니다.(불리언 인덱싱)
df.loc[df.index.month == 1]

In [ ]:
# df에서 매년 1월 2일인 행을 선택합니다.
df.loc[(df.index.month == 1) & (df.index.day == 2)]

In [ ]:
# df의 인덱스에서 정수 요일을 반환합니다.
# [참고] 정수 0은 Monday, 정수 6은 Sunday입니다.
df.index.dayofweek

In [ ]:
# df에서 매주 월요일인 행을 선택합니다.
df.loc[df.index.dayofweek == 0]

In [ ]:
# df의 인덱스에서 영문 요일 문자열을 반환합니다.
# [참고] date_format 매개변수에 지정한 날짜 포맷에 맞는 문자열을 반환합니다.
df.index.strftime(date_format='%A')

In [ ]:
# df의 인덱스에서 영문 요일 문자열을 반환합니다.
df.index.day_name()

In [ ]:
# 한글 요일 문자열을 매핑할 딕셔너리를 생성합니다.
weekday_kor = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}

In [ ]:
# df의 인덱스에서 정수 요일을 한글 요일명으로 매핑합니다.
df.index.dayofweek.map(mapper=weekday_kor)

### [참고] 집계 함수 활용법

In [ ]:
# df에서 매년 첫 번째 거래일인 행을 선택합니다.
# [참고] first 메서드는 그룹 집계 함수이므로, 그룹 키를 새로운 인덱스로 설정합니다.
df.groupby(by=df.index.year).first()

In [ ]:
# df에서 매년 마지막 거래일인 행을 선택합니다.
df.groupby(by=df.index.year).last()

In [ ]:
# df에서 매년 첫 번째 거래일인 행을 선택합니다.
# [참고] nth 메서드는 n번째 행을 선택하는 함수이므로, 원본 인덱스를 유지합니다.
df.groupby(by=df.index.year).nth(n=0)

In [ ]:
# df에서 매년 마지막 거래일인 행을 선택합니다.
df.groupby(by=df.index.year).nth(n=-1)

### 일간 수익률 계산

- 이번 예제에서 다루는 수익률은 투자 수익률이 아니고 종가 기준 가격 수익률입니다.
- 투자 수익률은 체결(진입/청산) 가격을 기준으로 계산해야 합니다.

In [ ]:
# 단순 수익률을 계산하고 df에 추가합니다.
# [참고] pct_change 메서드는 직전 값 대비 현재 값의 변화율을 반환합니다.
df['Simple_Return'] = df['Close'].pct_change()

In [ ]:
# 로그 수익률을 계산하고 df에 추가합니다.
# [참고] shift 메서드는 periods 매개변수에 지정한 정수만큼 위(음수) 또는 아래(양수)로 이동시킵니다.
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(periods=1))

In [ ]:
# df에서 열별 결측값 개수를 확인합니다.
df.isna().sum()

### 누적 수익률 계산

In [ ]:
# 누적 수익률을 계산하고 df에 추가합니다.
# [참고] 단순 수익률에 1을 더해 원금을 1로 설정합니다.
df['Cum_Return'] = (df['Simple_Return'] + 1).cumprod() - 1

In [ ]:
# df의 처음 10행을 확인합니다.
df.head(n=10)

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

In [ ]:
# 로그 수익률을 누적한 뒤 단순 누적 수익률로 변환합니다.
# [참고] 로그 누적 수익률을 지수 변환한 뒤, 1을 차감하면 단순 누적 수익률과 동일한 값이 됩니다.
np.exp(df['Log_Return'].cumsum()) - 1

### 누적 수익률 백분율 계산

In [ ]:
# 누적 수익률의 백분율을 계산하고 df에 추가합니다.
df['Cum_Return_Pct'] = df['Cum_Return'] * 100

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

In [ ]:
# 시작 종가와 마지막 종가를 각각 변수에 할당합니다.
base_close, last_close = df['Close'].iloc[[0, -1]]

In [ ]:
# 전체 기간 누적 수익률을 확인합니다.
(last_close / base_close - 1) * 100

### 변동성 계산

In [ ]:
# 단순 수익률의 평균을 확인합니다.
df['Simple_Return'].mean()

In [ ]:
# 단순 수익률의 표준편차를 확인합니다.
df['Simple_Return'].std()

In [ ]:
# 단순 수익률의 20일 이동 표준편차를 계산하고 df에 추가합니다.
df['Volatility_20'] = df['Simple_Return'].rolling(window=20).std()

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

### 이동 평균 계산

In [ ]:
# 종가의 20일 이동 평균을 계산하고 df에 추가합니다.
df['MA_20'] = df['Close'].rolling(window=20).mean()

In [ ]:
# 종가의 60일 이동 평균을 계산하고 df에 추가합니다.
df['MA_60'] = df['Close'].rolling(window=60).mean()

In [ ]:
# df의 처음 10행을 확인합니다.
df.head(n=10)

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

### Colab에서 한글 폰트 설치

In [ ]:
# 현재 Colab에 설치된 폰트 경로 목록을 확인합니다.
!fc-list

In [ ]:
# 설치된 전체 폰트의 패밀리 이름만 출력합니다.
!fc-list ':' family

In [ ]:
# Colab에서 사용할 나눔 폰트를 설치합니다.
!apt-get install -y fonts-nanum

In [ ]:
# 설치된 한글 폰트의 패밀리 이름을 중복 없이 오름차순 정렬하여 출력합니다.
!fc-list ':lang=ko' family | sort | uniq

### 한글 폰트명 탐색

In [ ]:
# 필요한 모듈을 임포트합니다.
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 설치된 폰트 파일 경로를 리스트로 생성합니다.
font_list = fm.findSystemFonts(fontext='ttf')

In [ ]:
# font_list의 원소 개수를 확인합니다.
len(font_list)

In [ ]:
# font_list에서 특정 폰트명을 포함하는 폰트 파일 경로를 선택하여 font_path에 할당합니다.
font_path = sorted([font for font in font_list if 'Nanum' in font])

In [ ]:
# font_path를 확인합니다.
font_path

In [ ]:
# 선택한 폰트 파일의 폰트명을 리스트로 확인합니다.
[fm.FontProperties(fname=font).get_name() for font in font_path]

In [ ]:
# matplotlib 라이브러리의 폰트 관리자 객체를 초기화합니다.
# [참고] 컴퓨터에 설치된 폰트 파일들을 다시 스캔하여 내부 폰트 캐시를 재구성하여 새로 설치한 한글 폰트를
# 사용할 수 있게 합니다.
fm.fontManager.__init__()

### 그래프 요소 설정

In [ ]:
# 한글 폰트와 글자 크기를 설정합니다.
plt.rc(group='font', family='NanumBarunGothic', size=10)

In [ ]:
# 그래프 크기와 해상도를 설정합니다.
plt.rc(group='figure', figsize=(8, 4), dpi=120)

In [ ]:
# 축에 유니코드 마이너스를 출력하지 않도록 설정합니다.
plt.rc(group='axes', unicode_minus=False)

In [ ]:
# 범례에 채우기 색과 테두리 색을 추가합니다.
plt.rc(group='legend', frameon=True, fc='0.9', ec='0.9')

### 종가 추이 선 그래프 시각화

In [ ]:
# 종가 추이를 선 그래프로 시각화합니다.
# [참고] color 매개변수에 0~1 범위의 실수를 문자열로 지정할 수 있습니다.
# '0'은 'black', '1'은 'white'이고, '0.5'는 'gray'입니다.
sns.lineplot(data=df, x=df.index, y='Close', color='0', lw=1)
plt.axhline(y=base_close, color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='주가(원)')
plt.title(label=f'{ticker} 종가(원)', fontweight='bold')
plt.show()

### 이동 평균 선 그래프 시각화

In [ ]:
# 종가와 이동 평균을 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Close', color='0.5', lw=1, label='종가')
sns.lineplot(data=df, x=df.index, y='MA_20', color='blue', lw=1, label='단기 이평')
sns.lineplot(data=df, x=df.index, y='MA_60', color='red', lw=1.5, label='중기 이평')
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='주가(원)')
plt.title(label=f'{ticker} 종가 및 이동 평균', fontweight='bold')
plt.show()

### 로그 수익률 선 그래프 시각화

In [ ]:
# 로그 수익률의 변화를 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Log_Return', color='red', lw=1)
plt.axhline(y=0, color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='로그 수익률')
plt.title(label=f'{ticker} 로그 수익률', fontweight='bold')
plt.show()

### 누적 수익률 선 그래프 시각화

In [ ]:
# 누적 수익률 추이를 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Cum_Return_Pct', color='red', lw=1)
plt.axhline(y=0, color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='백분율(%)')
plt.title(label=f'{ticker} 누적 수익률 백분율(%)', fontweight='bold')
plt.show()

### 로그 수익률 히스토그램 시각화

In [ ]:
# 로그 수익률의 기술통계량을 확인합니다.
df['Log_Return'].describe()

In [ ]:
# 로그 수익률의 분포를 히스토그램과 KDE로 시각화합니다.
# [참고] 로그 수익률은 정규분포처럼 보이지만 실제로는 양쪽 꼬리가 정규분포보다 두껍다는 특징을 가집니다.
sns.histplot(data=df, x='Log_Return', fc='0.8', ec='1',
             binrange=(-0.15, 0.15), bins=60, stat='density')
sns.kdeplot(data=df, x='Log_Return', color='red', lw=1.5)
plt.title(label=f'{ticker} 로그 수익률 히스토그램', fontweight='bold')
plt.show()

### 로그 수익률 Normal Q-Q Plot 시각화

In [ ]:
# 필요한 모듈을 임포트합니다.
from scipy import stats

In [ ]:
# 로그 수익률을 Normal Q-Q Plot으로 시각화합니다.
# [참고] 점들이 기준 직선에 대체로 가까우면 데이터가 정규분포에 가까운 것으로 판단합니다.
stats.probplot(x=df['Log_Return'].dropna(), dist='norm', plot=plt)
plt.xlabel(xlabel='정규분포 기준 이론적 분위수')
plt.ylabel(ylabel='표본 데이터 분위수')
plt.title(label=f'{ticker} Normal Q-Q Plot', fontweight='bold')
plt.show()

### 20일 이동 변동성 선 그래프 시각화

In [ ]:
# 20일 이동 변동성을 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Volatility_20', color='red', lw=1)
plt.axhline(y=df['Volatility_20'].mean(), color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='표준편차')
plt.title(label=f'{ticker} 20일 이동 표준편차', fontweight='bold')
plt.show()

## End of Document